In [52]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.utils import class_weight
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [53]:
# Load and clean dataset
data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(columns=['RowNumber', 'CustomerId', 'Surname'])


In [54]:
data.shape


(10000, 11)

In [55]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


In [56]:
# Check for null values
data.isnull().sum()


CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [57]:
data['Geography'].unique().sum()


'FranceSpainGermany'

In [58]:
ohe = OneHotEncoder()
geo_encoder = ohe.fit_transform(data[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=ohe.get_feature_names_out(['Geography']))


In [59]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)


In [60]:
# Save encoders
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
with open('ohe.pkl', 'wb') as file:
    pickle.dump(ohe, file)


In [61]:
# Split and scale
X = data.drop('Exited', axis=1)
y = data['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)


In [62]:
## Testing the problem
import pandas as pd

data = pd.read_csv('Churn_Modelling.csv')
print(data['Exited'].value_counts())

## Data is highly imbalanced


Exited
0    7963
1    2037
Name: count, dtype: int64


In [63]:
# Calculate class weights
class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))


In [64]:
# Build the model with dropout
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])


d:\AI python\ANN DL Project\ann_venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [65]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])


In [66]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [67]:
from sklearn.utils import class_weight
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
import numpy as np

# Step 1: Define class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}

# Step 2: Define callbacks
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Step 3: Train the model with class weights
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorflow_callback],
    class_weight=class_weights_dict
)


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6772 - loss: 0.6090 - val_accuracy: 0.8165 - val_loss: 0.4577
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7807 - loss: 0.5208 - val_accuracy: 0.7630 - val_loss: 0.5183
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7708 - loss: 0.5011 - val_accuracy: 0.8170 - val_loss: 0.4182
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7878 - loss: 0.4945 - val_accuracy: 0.8210 - val_loss: 0.4306
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7933 - loss: 0.4868 - val_accuracy: 0.8210 - val_loss: 0.4795
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8039 - loss: 0.4834 - val_accuracy: 0.8045 - val_loss: 0.4668
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7985 - loss: 0.4875 - val_accuracy: 0.7985 - val_loss: 0.4744
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8066 - loss: 0.4641 - val_accu

In [68]:
# Save the model
model.save('model.keras')


**Preparing Artificial Neural Networks**

In [69]:
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,837 (34.52 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,892 (23.02 KB)

In [70]:
## Load Tensorboard Extension
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [75]:
%tensorboard --logdir logs/fit/20250503-192252


Reusing TensorBoard on port 6008 (pid 10832), started 0:00:07 ago. (Use '!kill 10832' to kill it.)

In [74]:
  %reload_ext tensorboard


In [76]:
# Save the model
model.save('model.keras')


Issue Summary (Before Fixing)
Model was not trained properly:

The model had imbalanced data (only ~20% churn cases).

No class weighting was applied during training, so the model favored predicting "No Churn" (majority class).

Model overfit or underperformed:

Training accuracy was high (~87%) but validation accuracy dropped or fluctuated.

Predictions were always biased towards "not likely to churn", even for extreme churn-worthy inputs.

**TensorBoard showed early signs of accuracy stagnation and validation loss rising, hinting that the model was biased due to imbalance.

✅ Fixes We Applied
1. Introduced class_weight
We computed weights for each class (0 and 1) using:

python
Copy
Edit
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}
This told the model:

“Hey, class 1 (churners) is rarer, give it more importance when calculating loss.”

2. Re-trained with class_weight
python
Copy
Edit
model.fit(..., class_weight=class_weights_dict)
This improved:

Validation accuracy to ~85.4%

Validation loss stayed close to training loss

Predictions started becoming more meaningful

🎯 Final Outcome
Original problem: Model ignored churners due to class imbalance and insufficient generalization.

Fix applied: Used class weighting during training to make the model sensitive to rare churn cases.

Result: Balanced predictions, better validation accuracy, and stable loss.

1. Performance Metrics:
Accuracy: Ensure the accuracy of the model is high and stable. If accuracy starts to decrease, that’s a sign that the model might be overfitting, underfitting, or has become biased.

Loss: Watch the training and validation loss. The loss should ideally decrease and stabilize over time. If you notice that the validation loss starts to increase while the training loss continues to decrease (a sign of overfitting), you’ll need to take corrective measures.

Confusion Matrix: Check the confusion matrix (True Positives, False Positives, True Negatives, and False Negatives) periodically to see if the model is predicting both churn and non-churn customers correctly.

False Negatives: These occur when the model predicts a customer won’t churn but they actually do. This can be a problem in business contexts where you miss customers who need intervention.

False Positives: These occur when the model predicts a customer will churn but they don't. While less severe, it might mean wasting resources on the wrong customers.

2. Overfitting vs. Underfitting:
Overfitting: If the training accuracy is much higher than validation accuracy, the model might be overfitting (memorizing the training data). To prevent overfitting, you can try:

Early stopping.

Regularization (L2, dropout).

More data or augmentation.

Underfitting: If both training and validation accuracy are low, the model might be underfitting. To fix underfitting:

Try more complex models.

Increase the number of epochs.

Feature engineering.

3. Class Imbalance:
Since churn prediction typically has imbalanced classes (e.g., more customers do not churn than churn), it's essential to:

Monitor class distribution in your training, validation, and test datasets.

Adjust for class imbalance using techniques like class weights, oversampling, or undersampling.

Evaluate the model using metrics like Precision, Recall, and F1-score alongside accuracy, especially if the dataset is imbalanced.

4. Model Drift (Concept Drift):
Over time, customer behavior may change (e.g., due to market changes, new competitors, etc.), causing the model to lose accuracy. Monitor model drift by:

Periodically retraining the model on the most recent data.

Checking if predictions on recent data are still consistent with real outcomes.

Running A/B tests on your churn model to ensure its predictions align with real customer behavior.

5. Feature Engineering:
Always look at the features that are being used by the model.

Feature importance: Use techniques like SHAP or LIME to understand the most important features.

New features: As new data comes in, look for new patterns or features that could improve the model. If new relevant features are found, update your model.

Handling missing values: Make sure missing values are handled properly, and ensure the model continues to receive clean and consistent data.

6. Real-time Monitoring and Testing:
In a live application, keep an eye on predictions to catch any outliers or errors that could cause problems. This could involve:

Monitoring prediction trends.

Setting up alerts for anomalous behaviors (e.g., sudden drops in churn predictions).

Ensuring the model’s predictions align with business goals.

7. Model Interpretability:
Interpretability becomes crucial in real-world deployments. If your model is used by non-technical stakeholders (business teams, etc.), it's essential to have tools or techniques to explain why a particular customer is predicted to churn.

Tools like SHAP or LIME can provide insights into the important features contributing to each prediction.

8. Feedback Loop:
Collect feedback from the users of your churn prediction system. Are the predictions helpful? Do users agree with the predictions? A feedback loop can help improve the model over time, adjusting it based on how it is used in the real world.

9. Updating the Model:
Retrain periodically: The model might become outdated as more data comes in. Depending on how much data you're collecting, retrain the model regularly (e.g., quarterly or annually).

Version control: Track different versions of your model to keep track of changes and improvements over time. Use tools like MLflow or DVC for version control of models.

10. Infrastructure and Scalability:
Ensure that the model can scale with increasing amounts of data. If you're using the model in a production environment, ensure the infrastructure is set up for high availability and low latency.

Ensure the application serving predictions is stable and responsive.

Summary:
Monitor performance: accuracy, loss, confusion matrix, and other metrics.

Watch out for overfitting/underfitting and class imbalance.

Prevent model drift by retraining and keeping the model updated with fresh data.

Regularly review feature importance and refine features as needed.

Test and collect feedback to ensure predictions are aligned with business needs.

Update and retrain periodically with fresh data, and track model versions.

By keeping these factors in check, you'll be able to maintain a high-quality churn prediction model and adapt it as necessary over time.